# Antibody Aggregation Failure Pipeline
Run cells top to bottom.

In [ ]:
!git clone https://github.com/saptarshighosh10-oss/Biostuff.git
%cd Biostuff
!git checkout claude/antibody-aggregation-pipeline-xdx1s0
!pip install requests fair-esm -q

In [ ]:
# Basic run — BLOSUM62 mutations, no ESMFold (fast, ~3 min)
!python pipeline.py --entries 300 --variants 5 --mutations 2

In [ ]:
# Big run — 2000 PDB entries (antibodies + nanobodies + VHH + scFv + bispecifics)
# 20 variants each, ESM-2 guided, ESMFold on top 200 (~50-60 min)
# Parallel fetching keeps the network step to ~3 min
!python pipeline.py --entries 2000 --variants 20 --mutations 2 --esm2 --esmfold --top 200

In [ ]:
import json
import collections

with open('results/phase1_candidates.json') as f:
    candidates = json.load(f)

print(f'Total candidates saved: {len(candidates)}')

# --- Risk score distribution ---
risks = [c['risk']['combined_risk'] for c in candidates]
buckets = [('0.6+', 0.6, 1.1), ('0.5-0.6', 0.5, 0.6), ('0.4-0.5', 0.4, 0.5), ('<0.4', 0.0, 0.4)]
print('\nRisk score distribution:')
for label, lo, hi in buckets:
    n = sum(1 for r in risks if lo <= r < hi)
    print(f'  {label:8s} {"\u2588" * n} ({n})')

# --- Chain type breakdown ---
print('\nChain type breakdown:')
for t, n in collections.Counter(c['chain_type'] for c in candidates).most_common():
    print(f'  {t:8s}: {n}')

# --- Mutation hotspots ---
print('\nTop 10 mutated positions across all candidates:')
pos_counts = collections.Counter()
mut_counts = collections.Counter()
for c in candidates:
    for pos, orig, mut in c['mutations']:
        pos_counts[pos] += 1
        mut_counts[f'{orig}\u2192{mut}'] += 1
for pos, count in pos_counts.most_common(10):
    print(f'  position {pos:4d}: seen in {count} candidates')

print('\nTop 10 substitution types:')
for sub, count in mut_counts.most_common(10):
    print(f'  {sub}: {count}')

# --- Top 5 detailed ---
print('\n' + '='*60)
print('TOP 5 CANDIDATES (detailed)')
print('='*60)
for i, c in enumerate(candidates[:5]):
    r = c['risk']
    dscore = r.get('disagreement_score')
    dscore_str = f' | disagreement={dscore:.3f}' if dscore is not None else ''
    struct = r.get('structure_features', {})
    plddt_str = ''
    if struct:
        plddt_str = f' | mean_pLDDT={struct.get("mean_plddt", 0):.1f} | low_conf={struct.get("low_confidence_fraction", 0):.2f}'
    print(f'\n{i+1}. {c["anchor_pdb"]} ({c["chain_type"]})')
    print(f'   mutations:    {c["mutations"]}')
    print(f'   risk:         {r["combined_risk"]:.3f}{dscore_str}')
    print(f'   CamSol:       {r.get("camsol_score", 0):.3f}  (lower = worse solubility)')
    print(f'   hydrophobic:  {r.get("hydrophobicity_score", 0):.3f}{plddt_str}')


In [ ]:
from google.colab import files
files.download('results/phase1_candidates.json')